![Week 6 Day 1](../assets/banners/w6d1.png)

# Week 6 Day 1 - Production Architecture and the FastAPI Backend

Welcome to the final week! 🚢

Last Friday you measured GradeWise against a human examiner and found a baseline to beat. Look at what you have now: an exam graph that turns three PDFs into a validated `ExamPaper`, a knowledge base with hybrid search, a transcription subgraph that reads handwriting faithfully, a grader that cites evidence and resists prompt injection, report cards, a results agent, and an orchestrator that pauses for Emma whenever a mark is uncertain.

And yet Emma can't use any of it. She doesn't run Jupyter. She has a staff-room laptop, a browser and a pile of 30 scripts.

This week we **ship it**. Today we design the production architecture and put everything behind a **FastAPI** backend. You'll drive the whole API from this notebook (upload, run, status, review, reports) without starting a server and without spending a cent.

![Week 6 roadmap](../assets/roadmap/week6.png)

### 🎯 By the end of this lab you will
1. 🔁 Recap the five weeks of GradeWise, and see the one thing still missing
2. 🏗️ Sketch a production architecture: API, workers, UI and storage, and why each is separate
3. ⚡ Build a tiny FastAPI app from scratch: routes, Pydantic validation, `TestClient`
4. 🚀 Drive the real GradeWise API in 🧪 demo mode: health, uploads, a batch, polling, review and reports
5. 🧯 See how friendly errors and safe file names protect the API

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 1 - Week 6 Kickoff: Recap and Roadmap | 🎬 |
| 2 | Day 1 - Production Architecture: API, Workers, UI, Storage | 🎬 |
| 3 | Day 1 - FastAPI Backend: Upload, Run and Status Endpoints | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything this week runs <b>offline and free</b> in 🧪 <b>demo mode</b>: every model is swapped for a stand-in (<code>gradewise/api/demo.py</code>), so you exercise the real API, background jobs, interrupts and storage without an API key. The marks a demo run produces mean <b>nothing</b>: demo mode tests the plumbing, not the judgement. Today's files go to a sandbox, <code>output/week6/day1/</code>.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ No {key_name} set. That's fine this week: every cell runs offline in 🧪 demo mode.")
        print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

## Part 1: Where we are, and what's missing

Here's GradeWise after five weeks, in one table. Every row is code you wrote or read, living in the `gradewise/` package:

| Week | What GradeWise learned | Lives in |
|:-:|---|---|
| 1 | Turn a question paper + marking scheme into a validated `ExamPaper` (structured output) | `gradewise/schemas/`, `gradewise/exam/` |
| 2 | Ingest the CS201 notes into Chroma; hybrid search, reranking, retrieval metrics | `gradewise/ingestion/`, `gradewise/rag/` |
| 3 | Agents and graphs: `create_agent`, middleware, `StateGraph`, `Send`, `interrupt` | `gradewise/exam/graph.py` |
| 4 | Agentic RAG with evidence; faithful handwriting transcription | `gradewise/rag/agentic_rag.py`, `gradewise/transcription/` |
| 5 | Evidence-based grading, guardrails, report cards, the orchestrator, evaluation | `gradewise/grading/`, `gradewise/reports/`, `gradewise/orchestrator.py` |
| **6** | **An app Emma can use: API, dashboard, Docker, deployment** | `gradewise/api/`, `gradewise/ui/`, `docker/` |

The missing piece isn't intelligence any more. It's **access**: a way for a teacher (or any other program) to use GradeWise without Python.

## Part 2: Production architecture: API, workers, UI, storage

A notebook does one thing at a time, for one person, and forgets everything when the kernel restarts. A product has to do better on every one of those. Here's the shape we'll build this week:

![Diagram](../assets/diagrams/7e12e44774c0.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  subgraph UI["🖥️ UI · Streamlit (day 2)"]
    D[Dashboard pages]
  end
  subgraph API["🔌 API · FastAPI (today)"]
    R[Routes + Pydantic models]
    J[Job registry]
    S[SSE /events]
  end
  subgraph W["⚙️ Workers"]
    O[Orchestrator<br/>transcribe → grade → ⏸ review → report]
  end
  subgraph ST["💾 Storage · output/"]
    DB[(SQLite: exams, reports)]
    CK[(SQLite: checkpoints)]
    VS[(Chroma)]
    F[uploads + report cards]
  end
  D -- "HTTP + JSON" --> R
  D -- "SSE stream" --> S
  R --> J --> O
  O -- "on_event()" --> J --> S
  O --> DB & CK & VS & F
```

</details>


| Box | Its job | Why it's separate |
|---|---|---|
| 🖥️ **UI** | Forms, tables, the review queue | It only speaks HTTP. Swap Streamlit for React tomorrow and the backend doesn't notice |
| 🔌 **API** | Validate requests, start work, report status | Must answer in **milliseconds**, for many users at once |
| ⚙️ **Workers** | The slow part: vision, grading, report writing | Grading a class takes minutes; it must never block a web request |
| 💾 **Storage** | Exams, reports, checkpoints, vectors, files | Outlives every process: a paused review is still there tomorrow |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Imagine <code>POST /grade</code> simply ran the whole orchestrator for 30 students and returned the report cards at the end. Make a list of what goes wrong before you read on. (Think: browser timeouts, Emma refreshing the page, two teachers at once, and what happens at the first <code>interrupt()</code>.)</span>
        </td>
    </tr>
</table>

Here's the list most people come up with:

- ⏱️ **Timeouts.** Browsers, proxies and hosting platforms cut requests off after 30 to 100 seconds. A class takes minutes.
- 🔁 **Refresh = start again.** If the response is the only place the result lives, a refresh throws the work away and pays for it twice.
- 🧍🧍 **One at a time.** While one request grades, a busy server can't answer anyone else quickly.
- ⏸️ **Interrupts don't fit.** The orchestrator *pauses* for Emma. An HTTP request can't wait an hour for a human.

So the API does three cheap things instead: **start** a background job and return its ID at once (HTTP `202 Accepted`), report its **status** whenever asked, and **resume** it when Emma decides. That's exactly the shape of the endpoints you'll use in a moment.

## Part 3: FastAPI in five minutes

[FastAPI](https://fastapi.tiangolo.com/) turns ordinary Python functions into HTTP endpoints. Its superpower is that it reuses something you already know well: **Pydantic**. The type hints on your function become the validation, the JSON conversion *and* the interactive docs.

Let's build the smallest useful example: an endpoint that records a mark, and refuses a mark above the question's maximum (handbook principle 7, as an HTTP rule).

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

mini = FastAPI(title="Mini marker")


class Mark(BaseModel):
    question_id: str = Field(pattern=r"^Q\d+[a-z]?$")  # 💡 "Q4b" yes, "../etc" no
    awarded: float = Field(ge=0)
    max_marks: float = Field(gt=0)


@mini.get("/health")
def health() -> dict:
    return {"status": "ok"}


@mini.post("/marks", status_code=201)
def record_mark(mark: Mark) -> Mark:  # 💡 the JSON body is parsed and validated into a Mark for you
    if mark.awarded > mark.max_marks:
        raise HTTPException(status_code=422, detail=f"{mark.awarded} is more than the {mark.max_marks} available")
    return mark


mini_client = TestClient(mini)  # 💡 an HTTP client that calls the app directly: no server, no port
print(mini_client.get("/health").json())
print(mini_client.post("/marks", json={"question_id": "Q4b", "awarded": 2, "max_marks": 3}).json())

Now two bad requests. One breaks a **schema** rule (Pydantic catches it before your function even runs), the other breaks a **business** rule (your function catches it):

In [ ]:
for body in (
    {"question_id": "../secret", "awarded": 2, "max_marks": 3},  # ❌ fails the pattern
    {"question_id": "Q4b", "awarded": 5, "max_marks": 3},  # ❌ more than the maximum
):
    response = mini_client.post("/marks", json=body)
    print(response.status_code, str(response.json())[:160])

💡 Three FastAPI facts you'll see everywhere in `gradewise/api/main.py`:

1. **Plain `def` routes run in a thread pool**, so a slow one (an ingest, an LLM call) never freezes the server for everyone else. `async def` routes run on the event loop and must never block: GradeWise only uses `async def` for the SSE stream.
2. **`TestClient`** speaks real HTTP to your app in-process. It's how `tests/test_api.py` tests the whole API, and how we'll drive GradeWise today.
3. **`/docs`** is free: FastAPI builds an interactive OpenAPI page from your type hints. When you run the real server, open http://localhost:8000/docs and click around.

## Part 4: Meet the GradeWise API

Open `gradewise/api/main.py`. The banner at the top lists every endpoint. The app is built by one function:

```python
create_app(output_dir=None, *, demo=None) -> FastAPI
```

- `output_dir`: where everything is written (default `output/`). Demo mode writes to `<output_dir>/demo/`, so fake marks never mix with real ones.
- `demo`: the default mode for requests that don't say. Every route also accepts its own `demo=true|false`.

The cell below builds a private copy for this notebook, in demo mode, in a sandbox folder, plus two small helpers: `show()` to print JSON and `wait_for()` to poll a batch.

In [ ]:
import json
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day1"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

In [ ]:
# 💡 FastAPI describes itself as an OpenAPI document: the same data that powers the /docs page
paths = app.openapi()["paths"]
for path, operations in paths.items():
    for method, op in operations.items():
        print(f"{method.upper():<6} {path:<40} {op['tags'][0]:<16} {op['summary']}")

Fifteen routes, in five groups: **system**, **knowledge bases**, **exams**, **batches** and **reports** (+ chat). Let's call them in the order Emma would.

### `GET /health`: is it up, and which mode?

In [ ]:
show(client.get("/health").json())

Note `provider_key`: the dashboard uses it to switch demo mode on automatically when there's no key. And `jobs` counts batches by state, which is how a monitoring tool (or the dashboard's sidebar) knows something is waiting for a human.

### 🧪 What "demo mode" swaps

| Real step | In demo mode (`gradewise/api/demo.py`) |
|---|---|
| Exam extraction | `DemoExamModel` returns the bundled CS201 midterm. The **approval interrupt is real** |
| Handwriting | Synthetic students get their known transcript; any other PDF gets blank answers that pause for a human |
| Grading | `DemoGraderModel`: week 5's keyword examiner, "unsure" about half marks or less, so the review queue has work |
| Embeddings | `DeterministicFakeEmbedding` + BM25, so evidence citations are still real |
| Chat | `DemoResultsModel` picks one of the four results tools and shows its raw output |

## Part 5: Upload endpoints: the knowledge base and the exam

### 📚 `POST /knowledge-bases/{name}/documents`

Uploads arrive as **multipart form data**: the same encoding a browser uses for `<input type="file">`. Here we upload one of the CS201 PDFs ourselves, and ask the API to add the rest of the course notes from `data/` (the `include_course_notes` flag: all four PDFs, or just `unit1` and the handbook with `DATA_SIZE=small`). Behind the route is week 2's idempotent `ingest()`.

In [ ]:
pdf = DATA_DIR / "knowledge-base" / "unit1_operating_systems.pdf"
response = client.post(
    "/knowledge-bases/cs201/documents",
    files=[("files", (pdf.name, pdf.read_bytes(), "application/pdf"))],  # 💡 (field, (filename, bytes, type))
    data={"include_course_notes": "true"},  # 💡 form fields travel as strings
)
print(response.status_code)
show(response.json())

`201 Created`, 71 chunks with the full notes (fewer with `DATA_SIZE=small`), cost `$0.00` (fake embeddings). Run the cell again and watch `skipped`: unchanged files aren't re-embedded, a habit that saves real money in production.

### 📝 `POST /exams`: upload the exam pack, then approve

This is week 3's exam graph behind an endpoint. It extracts the exam and **pauses at the admin-review interrupt**, so the response is the exam *awaiting approval*, not a stored exam.

In [ ]:
pack = DATA_DIR / "exams" / "cs_midterm"
files = {
    key: (f"{key}.pdf", (pack / f"{key}.pdf").read_bytes(), "application/pdf")
    for key in ("question_paper", "marking_scheme", "methodology")
}

review = client.post("/exams", data={"exam_id": "cs_midterm"}, files=files).json()
print(
    f"{review['exam_id']}: {review['status']} · {review['summary']['question_count']} questions, "
    f"{review['summary']['total_marks']:g} marks · problems: {review['problems'] or 'none'}"
)

In [ ]:
approved = client.post("/exams/cs_midterm/approve", json={"approved": True}).json()
print(approved["status"], "·", approved["message"])
show([(e["exam_id"], e["source"]) for e in client.get("/exams").json()])

💡 The exam's LangGraph thread ID is deterministic (`exam-cs_midterm`) and checkpointed to SQLite, so Emma could upload on Monday, restart the server, and approve on Tuesday.

## Part 6: The run endpoint: `POST /batches`

Now the main event. `POST /batches` **starts** grading and returns at once with `202 Accepted` and a `batch_id`. Today we grade three synthetic students (two with `DATA_SIZE=small`) straight from their transcripts (`from_transcripts=true`, which skips the vision step: the cheapest way to run the real thing later, too).

In [ ]:
n_students = min(3, len(datasets.synthetic_students()))  # 💡 3, or 2 with DATA_SIZE=small

started = time.perf_counter()
created = client.post(
    "/batches", data={"exam_id": "cs_midterm", "from_transcripts": "true", "students": str(n_students)}
)
elapsed_ms = (time.perf_counter() - started) * 1000

print(f"HTTP {created.status_code} in {elapsed_ms:.0f} ms")
show(created.json())
batch_id = created.json()["batch_id"]

The request returned straight away (the first call also pays for some one-off imports); the grading is happening **right now** on a worker thread. Open `gradewise/api/jobs.py` to see how: a `BatchJob` dataclass holds the status, the progress and a list of numbered events, and a `JobRegistry` runs `grade_batch(...)` in a `ThreadPoolExecutor`.

## Part 7: The status endpoint: polling `GET /batches/{id}`

The simplest way to follow a job is to ask "how is it going?" every so often. Let's poll and print each student's progress:

In [ ]:
seen = None
while True:
    batch = client.get(f"/batches/{batch_id}").json()
    line = " · ".join(f"{p['student_id']} {p['stage']} {p['percent']}%" for p in batch["progress"])
    if line != seen:
        print(f"[{batch['status']:>7}] {line}")
        seen = line
    if batch["status"] in ("paused", "done", "failed"):
        break
    time.sleep(0.1)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The batch says <code>paused</code>, not <code>done</code>. Before running the next cell, predict: who paused it, and why? (Hint: week 5's guardrails, and what <code>DemoGraderModel</code> does with half marks.)</span>
        </td>
    </tr>
</table>

In [ ]:
pending = batch["pending"]
print(
    f"⏸️ {pending['type']} for {pending['student_id']}: {len(pending['items'])} item(s), "
    f"current total {pending['total']:g}/{pending['max_total']:g}\n"
)
item = pending["items"][0]
print(f"{item['question_id']} ({item['awarded']:g}/{item['max_marks']:g}): {item['question_text']}")
print(f"Why flagged : {item['reason']}")
print(f"Student wrote: {item['answer_text'][:200]}…")
for c in item["criteria"]:
    print(f"  {c['criterion_id']}: {c['awarded']:g}/{c['max_marks']:g} · {c['justification']} · {c['evidence']}")

That's the week 5 orchestrator's `human_review` interrupt, enriched by the API with everything Emma needs to decide: the question, what the student actually wrote, why it was flagged, and the grader's reasoning criterion by criterion, **with its evidence**. (Run `wait_for` + this cell later and you'll see that later students are flagged too.)

## Part 8: Review, resume, repeat

`POST /batches/{id}/review` answers the interrupt. For a `human_review` the body carries `overrides` (`{question_id: marks}`); for a transcription check it carries `corrections` (`{question_id: text}`). The API resumes the graph with `Command(resume=...)` on a worker thread and returns at once.

First, a safety check: the optional `student_id` must match the pending one, so two teachers can't approve the wrong script by accident:

In [ ]:
wrong = client.post(f"/batches/{batch_id}/review", json={"student_id": "student_99", "approved": True})
print(wrong.status_code, wrong.json()["error"], "·", wrong.json()["what"])

Now let's play Emma. She reads the first flagged answer, decides it deserves **full marks**, and approves everyone else's flags as they are:

In [ ]:
overridden = None
while batch["status"] == "paused":
    pending = batch["pending"]
    overrides = {}
    if overridden is None and pending["type"] == "human_review":
        first = pending["items"][0]
        overrides = {first["question_id"]: first["max_marks"]}  # 🧑‍🏫 Emma's decision
        overridden = (pending["student_id"], first["question_id"])
    body = {"student_id": pending["student_id"], "overrides": overrides, "approved": True}
    client.post(f"/batches/{batch_id}/review", json=body).raise_for_status()
    print(f"✅ reviewed {pending['student_id']} ({len(pending['items'])} flag(s), overrides={overrides})")
    batch = wait_for(batch_id)

print(f"\nBatch {batch['status']} · {batch['events']} events")
for r in batch["reports"]:
    print(f"  {r['student_id']}: {r['total']:g}/{r['max_total']:g} ({r['percentage']:g}%) band {r['grade_band']}")

## Part 9: The reports endpoints

`GET /reports/{exam_id}` is the class overview (computed in code, never by a model), and `GET /reports/{exam_id}/{student_id}` is one report card, as JSON, `?format=html` or `?format=pdf`:

In [ ]:
overview = client.get("/reports/cs_midterm").json()
print(
    f"{overview['count']} students · mean {overview['mean_percent']}% · median {overview['median_percent']}% · "
    f"bands {overview['bands']} · still flagged {overview['flagged']}"
)
print("Hardest questions:", [(q["question_id"], q["mean_percent"]) for q in overview["questions"][:3]])

sid, qid = overridden
card = client.get(f"/reports/cs_midterm/{sid}").json()
result = next(r for r in card["results"] if r["question_id"] == qid)
print(f"\n{sid} {qid}: {result['awarded_marks']:g}/{result['max_marks']:g} · {result['review_reason']}")

Emma's override is in the stored report card, **with an audit trail**: `review_reason` records that a teacher changed it, and from what. That matters on Day 5, when we talk about transparency.

## Part 10: Friendly errors, safe names

An API is a public front door, so two habits from `main.py` are worth copying into every project.

**1. Errors have the same four friendly fields as in the terminal.** `create_app()` registers an exception handler that turns any `GradeWiseError` into JSON with `error`, `what`, `cause`, `fix` and `help`, and the right HTTP status (`status_for()` maps not-found to 404, "not waiting" to 409, and so on). Unexpected exceptions become a 500 **without** a traceback: the full traceback stays in the server's terminal.

In [ ]:
for method, path, kwargs in [
    ("get", "/batches/b-nope", {}),
    ("post", "/batches", {"data": {"exam_id": "cs_midterm"}}),  # no answer sheets given
    (
        "post",
        "/exams",
        {
            "data": {"exam_id": "x"},
            "files": {
                k: (f"{k}.pdf", b"hello", "application/pdf")
                for k in ("question_paper", "marking_scheme", "methodology")
            },
        },
    ),
    ("get", "/reports/cs_midterm/..secret", {}),  # a name that could be a path trick
]:
    r = getattr(client, method)(path, **kwargs)
    body = r.json()
    print(f"{r.status_code} {method.upper():<4} {path}")
    print(f"    {body.get('error')}: {body.get('what', body)}\n    fix: {body.get('fix')}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Never build a path from raw user input.</b> A student ID like <code>../../secret</code> would escape the reports folder. <code>gradewise/api/workspace.py</code> allow-lists names with <code>safe_name()</code> (letters, digits, <code>-</code> and <code>_</code> only) and strips uploaded file names down with <code>safe_filename()</code>. And <code>save_upload()</code> checks that a "PDF" really starts with <code>%PDF-</code>: the browser's word isn't proof. Uploads always land in <code>output/uploads/</code>, never in <code>data/</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">An API turns GradeWise from a demo into a <b>platform</b>. The same endpoints serve Emma's dashboard, a school's learning-management system (Moodle, Canvas) through a small integration, a nightly batch job, or another team's app. That's how EdTech products sell: one backend, many front doors. And because every request can say <code>demo=true</code>, sales demos and CI pipelines run the whole product for free.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Everything you called today lives in <code>gradewise/api/</code>: <code>main.py</code> (the routes and <code>create_app()</code>), <code>schemas.py</code> (request/response models), <code>jobs.py</code> (background batches and their events: read its 🚀 PRO note on Celery, Redis and Postgres), <code>workspace.py</code> (where files go) and <code>demo.py</code> (the fakes). Run the real server with <code>uv run python -m gradewise.api --demo</code> (or <code>uv run python -m gradewise serve api --demo</code>) and open <a href="http://localhost:8000/docs">localhost:8000/docs</a>. The tests that drive it exactly like this notebook are in <code>tests/test_api.py</code>.</span>
        </td>
    </tr>
</table>

Want to see the same flow from a terminal? With the server running in another terminal:

```bash
curl localhost:8000/health
curl -X POST localhost:8000/batches -F exam_id=cs_midterm -F from_transcripts=true -F students=3 -F demo=true
curl localhost:8000/batches/<batch_id>
curl -X POST localhost:8000/batches/<batch_id>/review -H "Content-Type: application/json" -d '{"approved": true}'
```

> ⚠️ **Windows:** always start servers with `uv run python -m ...` (not `uv run uvicorn ...`): locked-down machines can block freshly created `.exe` launchers, while `python -m` always works.

## Recap, and where we are heading

Today you:
- 🔁 recapped five weeks of GradeWise and named the missing piece: access
- 🏗️ split the product into **UI, API, workers and storage**, and saw why grading can't run inside a request
- ⚡ built a mini FastAPI app with Pydantic validation and tested it with `TestClient`
- 🚀 drove the real API in demo mode: health → knowledge base upload → exam upload and approval → `202` batch → polling → review with an override → reports
- 🧯 saw friendly errors, 409s for stale reviews, and safe names and uploads

**Tomorrow (Day 2):** Emma gets a face for all this: the **Streamlit dashboard**, and the page that matters most, the **review queue**.

⏭ **Next:** `week6/day2.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Add your own endpoint. Create an <code>APIRouter</code> with <code>GET /leaderboard/{exam_id}?n=3</code> that returns the top <code>n</code> students (ID, percentage, band) with a Pydantic response model, include it in the <code>app</code> from this notebook with <code>app.include_router(...)</code>, and test it with <code>client</code>. Bonus: when there are no reports, raise a <code>GradeWiseError</code> subclass so the friendly error handler answers with a 404. Hints: <code>gradewise.db.list_reports(exam_id, db=...)</code>, and the demo workspace is <code>app.state.workspaces[True]</code>. Solution: <code>week6/solutions/day1_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• FastAPI tutorial: <a href="https://fastapi.tiangolo.com/tutorial/">fastapi.tiangolo.com/tutorial</a><br/>• Testing with <code>TestClient</code>: <a href="https://fastapi.tiangolo.com/tutorial/testing/">fastapi.tiangolo.com/tutorial/testing</a><br/>• Request files and forms: <a href="https://fastapi.tiangolo.com/tutorial/request-forms-and-files/">fastapi.tiangolo.com/tutorial/request-forms-and-files</a><br/>• HTTP status codes, explained: <a href="https://developer.mozilla.org/en-US/docs/Web/HTTP/Status">MDN: HTTP response status codes</a></span>
        </td>
    </tr>
</table>